# Code Comment Generation — Fine-tuning CodeT5 on CodeSearchNet

**Goal:** fine-tune `Salesforce/codet5-small` to generate natural-language
comments/docstrings from source code, using a filtered subset of
`sentence-transformers/codesearchnet`.

**Important dataset fact (verified before writing this notebook):**
this HF dataset config (`"pair"`) has **only a `train` split** (1,375,067 rows)
and **two columns**: `comment` (str) and `code` (str). There is no
validation/test split shipped with it — we build our own below.

**Phases in this notebook:**
1. Install + load dataset, inspect schema
2. Filter low-quality pairs, build train/val/test splits
3. Load CodeT5-small, run a BEFORE-fine-tuning baseline
4. Tokenize, fine-tune with `Seq2SeqTrainer`
5. Evaluate (BLEU / ROUGE-L / exact match) on held-out test set, compare to baseline
6. Save qualitative samples + push model to the Hugging Face Hub

Run top-to-bottom on a Kaggle GPU (T4 x1 is enough for `codet5-small` on a ~10k-example subset).


**Dependency pin — read before running:** `transformers` released a full
**v5.0.0** in early 2026 with a complete tokenizer rewrite (new
`TokenizersBackend` / `PythonBackend` system). CodeT5's tokenizer predates
this — it's an old Roberta-style tokenizer (`vocab.json` + `merges.txt`, no
`tokenizer.json`) — and v5's loading path currently has a regression that
breaks on exactly that legacy format, raising:

```
TypeError: extra_special_tokens must be a list/tuple of str or AddedToken, or a dict mapping names to tokens
```

`pip install transformers` with no version pin installs v5 by default now,
so we pin to the last stable v4 release below. If a future `transformers`
patch fixes this for older tokenizers, you can drop the pin — but pinning
is also just good practice for a portfolio project: it keeps this notebook
reproducible regardless of what ships on PyPI next week.

In [ ]:
# Install Libraries
# Pinned to the last stable v4 release -- transformers v5's tokenizer rewrite
# currently breaks loading CodeT5's legacy (Roberta-style) tokenizer files.
# huggingface_hub is pinned below 1.0 to match what transformers v4.x expects
# (v5 requires huggingface_hub>=1.0, which would otherwise get pulled in).
!pip install -q "transformers==4.57.3" "huggingface_hub<1.0" datasets evaluate sentencepiece accelerate

In [ ]:
import os
import json
import random

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
import evaluate

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 1. Load the dataset and inspect its real schema

In [ ]:
dataset = load_dataset("sentence-transformers/codesearchnet", "pair")
print(dataset)
print()
sample = dataset["train"][0]
for key, value in sample.items():
    print(f"\n{key}:")
    print(value)

## 2. Filter low-quality pairs

Drop pairs with degenerate code/comment lengths, boilerplate comments (`TODO`, `FIXME`, ...), and comments that are mostly punctuation noise.

In [ ]:
MIN_CODE_CHARS, MAX_CODE_CHARS = 20, 4000
MIN_COMMENT_CHARS, MAX_COMMENT_CHARS = 8, 400
BOILERPLATE_PREFIXES = ("todo", "fixme", "xxx", "deprecated")


def is_valid_pair(example):
    code_str = example.get("code") or ""
    comment_str = example.get("comment") or ""

    if not (MIN_CODE_CHARS <= len(code_str) <= MAX_CODE_CHARS):
        return False
    if not (MIN_COMMENT_CHARS <= len(comment_str) <= MAX_COMMENT_CHARS):
        return False
    if comment_str.strip().lower().startswith(BOILERPLATE_PREFIXES):
        return False

    alnum_chars = sum(c.isalnum() for c in comment_str)
    if alnum_chars < 0.3 * len(comment_str):
        return False

    return True


raw_train = dataset["train"]
print("Before filtering:", len(raw_train))

filtered = raw_train.filter(is_valid_pair, num_proc=4)
print("After filtering: ", len(filtered))

## 3. Build train / val / test splits

Subset sizes below are tuned for a **fast** run that still gives a credible before/after comparison. Bump `TRAIN_SIZE` up later for a stronger model once the pipeline is verified end-to-end.

In [ ]:
TRAIN_SIZE = 8000
VAL_SIZE = 1000
TEST_SIZE = 1000

shuffled = filtered.shuffle(seed=SEED)
total_needed = TRAIN_SIZE + VAL_SIZE + TEST_SIZE
assert len(shuffled) >= total_needed, "Filtered dataset is smaller than requested subset!"

subset = shuffled.select(range(total_needed))
train_ds = subset.select(range(0, TRAIN_SIZE))
val_ds = subset.select(range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE))
test_ds = subset.select(range(TRAIN_SIZE + VAL_SIZE, TRAIN_SIZE + VAL_SIZE + TEST_SIZE))

print("train:", len(train_ds), "val:", len(val_ds), "test:", len(test_ds))

## 4. Load CodeT5-small

In [ ]:
MODEL_NAME = "Salesforce/codet5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Model on:", device)

## 5. Baseline — BEFORE fine-tuning

This is the comparison point that makes the CV story concrete: "fine-tuning improved BLEU from X to Y".

In [ ]:
def generate_comment(code_str, model, tokenizer, max_input_len=256, max_output_len=64, num_beams=4):
    inputs = tokenizer(code_str, return_tensors="pt", max_length=max_input_len, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_length=max_output_len, num_beams=num_beams)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


print("--- BEFORE fine-tuning (pretrained codet5-small) ---\n")
for i in range(3):
    code_str = test_ds[i]["code"]
    ref = test_ds[i]["comment"]
    pred = generate_comment(code_str, model, tokenizer)
    print("CODE:     ", code_str[:200].replace("\n", " "))
    print("REFERENCE:", ref)
    print("PREDICTED:", pred)
    print()

## 6. Tokenize for training

Pad-token positions in the labels are set to `-100` so the loss ignores them (standard seq2seq practice).

In [ ]:
MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 64


def preprocess(batch):
    model_inputs = tokenizer(
        batch["code"],
        max_length=MAX_SOURCE_LEN,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        text_target=batch["comment"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length",
    )
    label_ids = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in ids]
        for ids in labels["input_ids"]
    ]
    model_inputs["labels"] = label_ids
    return model_inputs


train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)
test_tok = test_ds.map(preprocess, batched=True, remove_columns=test_ds.column_names)

print(train_tok)

## 7. Metrics for the Trainer's eval loop (BLEU + ROUGE-L)

In [ ]:
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    bleu = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels],
    )
    rouge = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "bleu": round(bleu["bleu"] * 100, 2),
        "rougeL": round(rouge["rougeL"] * 100, 2),
    }

## 8. Fine-tune with `Seq2SeqTrainer`

**Note on `transformers` API:** this notebook uses `eval_strategy` (not the deprecated `evaluation_strategy`) and `processing_class` (not the deprecated `tokenizer=` kwarg) — the current API as of the transformers version installed by the pip install cell above.

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./codet5-comment-gen",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

## 9. Final evaluation on the held-out TEST set

We evaluate on `test_ds`, which the trainer never saw during training *or* validation-based checkpoint selection.

In [ ]:
def batch_generate(gen_model, tok, codes, max_input_len=256, max_output_len=64, num_beams=4, batch_size=16):
    gen_model.eval()
    preds = []
    for i in range(0, len(codes), batch_size):
        batch = codes[i : i + batch_size]
        inputs = tok(
            batch,
            return_tensors="pt",
            max_length=max_input_len,
            truncation=True,
            padding=True,
        ).to(gen_model.device)
        with torch.no_grad():
            outputs = gen_model.generate(**inputs, max_length=max_output_len, num_beams=num_beams)
        preds.extend(tok.batch_decode(outputs, skip_special_tokens=True))
    return preds


def exact_match(preds, refs):
    if not preds:
        return 0.0
    matches = sum(p.strip().lower() == r.strip().lower() for p, r in zip(preds, refs))
    return matches / len(preds)


test_codes = test_ds["code"]
test_refs = test_ds["comment"]

test_preds_finetuned = batch_generate(model, tokenizer, test_codes)

final_bleu = bleu_metric.compute(predictions=test_preds_finetuned, references=[[r] for r in test_refs])
final_rouge = rouge_metric.compute(predictions=test_preds_finetuned, references=test_refs)
final_exact = exact_match(test_preds_finetuned, test_refs)

finetuned_results = {
    "bleu": round(final_bleu["bleu"] * 100, 2),
    "rouge1": round(final_rouge["rouge1"] * 100, 2),
    "rouge2": round(final_rouge["rouge2"] * 100, 2),
    "rougeL": round(final_rouge["rougeL"] * 100, 2),
    "exact_match": round(final_exact * 100, 2),
    "n_test_examples": len(test_refs),
}
print("FINE-TUNED model on held-out test set:")
print(json.dumps(finetuned_results, indent=2))

with open("test_metrics_finetuned.json", "w") as f:
    json.dump(finetuned_results, f, indent=2)

## 10. Baseline comparison on the SAME test set

Re-load the original pretrained weights (your fine-tuned `model` variable has already been updated in place) so the before/after comparison is apples-to-apples.

In [ ]:
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
test_preds_baseline = batch_generate(baseline_model, tokenizer, test_codes)

baseline_bleu = bleu_metric.compute(predictions=test_preds_baseline, references=[[r] for r in test_refs])
baseline_rouge = rouge_metric.compute(predictions=test_preds_baseline, references=test_refs)
baseline_exact = exact_match(test_preds_baseline, test_refs)

baseline_results = {
    "bleu": round(baseline_bleu["bleu"] * 100, 2),
    "rouge1": round(baseline_rouge["rouge1"] * 100, 2),
    "rouge2": round(baseline_rouge["rouge2"] * 100, 2),
    "rougeL": round(baseline_rouge["rougeL"] * 100, 2),
    "exact_match": round(baseline_exact * 100, 2),
    "n_test_examples": len(test_refs),
}

print("BASELINE (pretrained, no fine-tuning) on the SAME held-out test set:")
print(json.dumps(baseline_results, indent=2))
print()
print("FINE-TUNED:")
print(json.dumps(finetuned_results, indent=2))

with open("test_metrics_baseline.json", "w") as f:
    json.dump(baseline_results, f, indent=2)

## 11. Qualitative samples (for the README)

In [ ]:
def format_samples_markdown(codes, preds, refs, n=10):
    lines = ["| Code (truncated) | Predicted | Reference |", "|---|---|---|"]
    for c, p, r in list(zip(codes, preds, refs))[:n]:
        c_clean = c.strip().replace("\n", " ").replace("|", "\\|")[:150]
        p_clean = p.strip().replace("|", "\\|")
        r_clean = r.strip().replace("|", "\\|")
        lines.append(f"| `{c_clean}` | {p_clean} | {r_clean} |")
    return "\n".join(lines)


samples_md = format_samples_markdown(test_codes, test_preds_finetuned, test_refs, n=10)
print(samples_md)

with open("qualitative_samples.md", "w") as f:
    f.write(samples_md)

## 12. Save the model (and optionally push to the Hugging Face Hub)

Pushing to the Hub is the easiest way to get the model into the FastAPI demo app later — `from_pretrained("your-username/...")` just works from anywhere with internet access.

In [ ]:
model.save_pretrained("./codet5-comment-gen-final")
tokenizer.save_pretrained("./codet5-comment-gen-final")
print("Saved locally to ./codet5-comment-gen-final")

# --- Uncomment to push to the Hub ---
# from huggingface_hub import login
# login(token="hf_...")  # or use a Kaggle secret instead of pasting the token directly
#
# MODEL_REPO_ID = "your-username/codet5-small-comment-generator"  # <-- change this
# model.push_to_hub(MODEL_REPO_ID)
# tokenizer.push_to_hub(MODEL_REPO_ID)

## Next steps

1. Download `test_metrics_finetuned.json`, `test_metrics_baseline.json`, and
   `qualitative_samples.md` from the Kaggle output and drop them into the
   `results/` folder of the GitHub repo.
2. Push the model to the Hugging Face Hub (uncomment the cell above), then
   set `MODEL_NAME` in `app/main.py` to your Hub repo id.
3. Run the FastAPI demo locally to try real inference end-to-end.
4. Once the pipeline above is verified, consider raising `TRAIN_SIZE` for a
   stronger model — the filtering/splitting/tokenization code does not need
   to change.
